In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import numpy as np
import os

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True) 
        print("GPU detected and configured with memory growth")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected. Falling back to CPU.")

TARGET_H = 48 
TARGET_W = 48
N_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 20
INITIAL_LR = 1e-4

In [ ]:
print("Loading Fashion MNIST dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

def preprocess_image(image, label):
    """Applies resizing, normalization, and grayscale-to-RGB conversion."""
    image = tf.cast(image, tf.float32) / 255.0
    image_resized = tf.image.resize(tf.expand_dims(image, axis=-1), (TARGET_H, TARGET_W))
    image_rgb = tf.image.grayscale_to_rgb(image_resized)
    label_one_hot = tf.one_hot(label, N_CLASSES)
    return image_rgb, label_one_hot

def augment_image(image, label):
    """Applies data augmentation to the training images."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.rot90(image, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    image = tf.image.random_brightness(image, max_delta=0.1)
    return image, label

In [ ]:
print("\nCreating tf.data pipeline with augmentation")

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = test_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_images_norm = np.array([preprocess_image(img, 0)[0].numpy() for img in x_test])
test_labels_one_hot = tf.one_hot(y_test, N_CLASSES)

In [ ]:
print("\nBuilding VGG16 model (Revision 3: block4/5 fine-tuned, Dropout 0.5)")

vgg = VGG16(
    weights='imagenet', 
    include_top=False, 
    input_shape=(TARGET_H, TARGET_W, 3)
)

for layer in vgg.layers:
    layer.trainable = False
    
for layer in vgg.layers:
    if layer.name.startswith('block4') or layer.name.startswith('block5'):
        layer.trainable = True

x = vgg.output
x = Flatten()(x)

x = Dense(512)(x) 
x = BatchNormalization()(x) 
x = tf.keras.activations.relu(x)

x = Dropout(0.5)(x) 

predictions = Dense(N_CLASSES, activation='softmax')(x)

model = Model(inputs=vgg.input, outputs=predictions)

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.5, 
    patience=3, 
    min_lr=1e-7, 
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=6,
    restore_best_weights=True,
    verbose=1
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LR), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
print(f"\n🚀 Starting training for up to {EPOCHS} epochs (Training base for TTA)...")

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[reduce_lr, early_stopping]
)

print("\nTraining complete.")

MODEL_SAVE_PATH = 'vgg16_fmnist_9367_tta.keras'

print(f"\nSaving the best model weights to {MODEL_SAVE_PATH}...")
try:
    model.save(MODEL_SAVE_PATH)
    print("Model successfully saved!")
except Exception as e:
    print(f"Error saving model: {e}")

In [ ]:
def apply_tta(model, images, labels_one_hot, n_views=8):
    """Performs Test-Time Augmentation (TTA) prediction."""
    all_predictions = []

    for i in range(n_views):
        augmented_batch = []
        for img in images:
            img_aug = tf.image.random_flip_left_right(img) 
            img_aug = tf.image.rot90(img_aug, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)) 
            img_aug = tf.image.random_brightness(img_aug, max_delta=0.1)
            augmented_batch.append(img_aug.numpy())

        augmented_batch = np.array(augmented_batch)
        
        predictions = model.predict(augmented_batch, batch_size=BATCH_SIZE, verbose=0)
        all_predictions.append(predictions)

    avg_predictions = np.mean(all_predictions, axis=0)
    
    predicted_classes = np.argmax(avg_predictions, axis=1)
    true_classes = np.argmax(labels_one_hot, axis=1)
    
    tta_accuracy = np.mean(predicted_classes == true_classes)
    return tta_accuracy

print("\nRunning FINAL Test-Time Augmentation (TTA) Evaluation")

loss, accuracy = model.evaluate(test_dataset, verbose=0)
print(f"Test Accuracy (Standard, No TTA): {accuracy:.4f}")

final_tta_accuracy = apply_tta(model, test_images_norm, test_labels_one_hot, n_views=8) 

print(f"**Test Accuracy (FINAL, 8-View TTA): {final_tta_accuracy:.4f}**")